# Construction du dataset CropYield Prediction

Assemblage des quatre fichiers sources en un dataset unique à la granularité
**pays × année × culture**, clé `iso3 + year + crop`, sur **1990-2013**.


| Décision | Motif |
|---|---|
| Période 1990-2013 | intersection des quatre sources |
| Jointure sur `iso3` | la jointure par nom littéral perd 18 pays, dont États-Unis, Chine, Russie |
| `China, mainland` retenu | les trois sources explicatives couvrent la Chine continentale |
| `temp.csv` dédoublonné puis moyenné | plusieurs relevés par couple pays/année, sinon fan-out |
| `rain_mm` conservée sans imputation | normale climatique par pays, constante dans le temps |

# Imports

In [1]:
import pandas as pd

from agritech import geo
from agritech.config import PATHS

# Sources

In [2]:
DEBUT, FIN = 1990, 2013
files_dir = PATHS.data_crop_yield_prediction

df_yield = pd.read_csv(files_dir / "yield.csv")
df_temp = pd.read_csv(files_dir / "temp.csv")
df_rainfall = pd.read_csv(files_dir / "rainfall.csv")
df_pesticides = pd.read_csv(files_dir / "pesticides.csv")

# rainfall.csv a une espace initiale dans l'en-tête de sa colonne pays
df_rainfall.columns = df_rainfall.columns.str.strip()

for nom, df in [("yield", df_yield), ("temp", df_temp),
                ("rainfall", df_rainfall), ("pesticides", df_pesticides)]:
    print(f"{nom:12} {df.shape[0]:6} lignes x {df.shape[1]} colonnes")

yield         56717 lignes x 12 colonnes
temp          71311 lignes x 3 colonnes
rainfall       6727 lignes x 3 colonnes
pesticides     4349 lignes x 7 colonnes


# Préparation des sources

In [3]:
def ajoute_iso3(df, colonne_pays):
    """Ajoute une colonne iso3 à partir du nom de pays."""
    correspondances = geo.vers_iso3(df[colonne_pays].astype(str).unique())
    return df.assign(iso3=df[colonne_pays].map(correspondances))

## Rendement

`China` est l'agrégat incluant Taïwan, Hong Kong et Macao ; `China, mainland` est la
Chine continentale seule. Les trois sources explicatives isolent ces territoires, on
retient donc `China, mainland` pour joindre des périmètres comparables.

In [4]:
rendement = (
    df_yield[df_yield["Area"] != "China"]
    .query("@DEBUT <= Year <= @FIN")
    .pipe(ajoute_iso3, "Area")
    .rename(columns={"Area": "area", "Year": "year", "Item": "crop"})
    .assign(yield_t_ha=lambda d: d["Value"] / 10_000)   # hg/ha -> t/ha
    [["iso3", "area", "year", "crop", "yield_t_ha"]]
)

rendement.head()

,iso3,area,year,crop,yield_t_ha
29,AFG,Afghanistan,1990,Maize,1.7582
30,AFG,Afghanistan,1991,Maize,1.6800
31,AFG,Afghanistan,1992,Maize,1.5000
32,AFG,Afghanistan,1993,Maize,1.6786
33,AFG,Afghanistan,1994,Maize,1.6667


## Température

Doublons stricts retirés, puis moyenne des relevés disponibles pour un pays et une
année, sans pondération géographique.

In [5]:
temperature = (
    df_temp.drop_duplicates()
    .query("@DEBUT <= year <= @FIN")
    .pipe(ajoute_iso3, "country")
    .groupby(["iso3", "year"], as_index=False)["avg_temp"].mean()
)

temperature.head()

,iso3,year,avg_temp
0,AFG,1990,15.45
1,AFG,1991,14.57
2,AFG,1992,14.35
3,AFG,1993,14.96
4,AFG,1994,14.94


## Pluie

`average_rain_fall_mm_per_year` est lue en `object` : quelques valeurs `..` bloquent la
conversion. Elles deviennent `NaN`, sans imputation.

In [6]:
pluie = (
    df_rainfall
    .assign(rain_mm=lambda d: pd.to_numeric(d["average_rain_fall_mm_per_year"], errors="coerce"))
    .query("@DEBUT <= Year <= @FIN")
    .pipe(ajoute_iso3, "Area")
    .rename(columns={"Year": "year"})
    [["iso3", "year", "rain_mm"]]
)

pluie.head()

,iso3,year,rain_mm
4,AFG,1990,327.0
5,AFG,1991,327.0
6,AFG,1992,327.0
7,AFG,1993,327.0
8,AFG,1994,327.0


## Pesticides

Une seule ligne par pays et par année, aucune agrégation nécessaire.

In [7]:
pesticides = (
    df_pesticides
    .query("@DEBUT <= Year <= @FIN")
    .pipe(ajoute_iso3, "Area")
    .rename(columns={"Year": "year", "Value": "pesticides_t"})
    [["iso3", "year", "pesticides_t"]]
)

pesticides.head()

,iso3,year,pesticides_t
0,ALB,1990,121.0
1,ALB,1991,121.0
2,ALB,1992,121.0
3,ALB,1993,121.0
4,ALB,1994,201.0


# Contrôles avant fusion

In [8]:
sources = {
    "rendement": (rendement, ["iso3", "year", "crop"]),
    "température": (temperature, ["iso3", "year"]),
    "pluie": (pluie, ["iso3", "year"]),
    "pesticides": (pesticides, ["iso3", "year"]),
}

for nom, (df, cle) in sources.items():
    sans_iso3 = df["iso3"].isna().sum()
    valides = df.dropna(subset=["iso3"])
    doublons = valides.duplicated(cle).sum()
    print(f"{nom:12} {len(df):6} lignes | {valides['iso3'].nunique():3} pays "
          f"| sans ISO3 : {sans_iso3:5} | doublons sur {'+'.join(cle)} : {doublons}")
    assert doublons == 0, f"{nom} : la clé {cle} n'est pas unique"

rendement     25848 lignes | 168 pays | sans ISO3 :  3169 | doublons sur iso3+year+crop : 0
température    3192 lignes | 133 pays | sans ISO3 :     0 | doublons sur iso3+year : 0
pluie          4991 lignes | 170 pays | sans ISO3 :  1081 | doublons sur iso3+year : 0
pesticides     3860 lignes | 144 pays | sans ISO3 :   506 | doublons sur iso3+year : 0


**Observations :**

- Les quatre sources sont uniques sur leur clé attendue : aucune agrégation supplémentaire n'est nécessaire.
- Les lignes sans code ISO3 correspondent aux micro-États, territoires insulaires et entités historiques ; elles ne sont présentes dans aucune combinaison exploitable des quatre sources.

# Fusion

In [9]:
dataset = rendement.dropna(subset=["iso3"])
print(f"base rendement  {len(dataset)} lignes")

for nom, source in [("température", temperature), ("pluie", pluie), ("pesticides", pesticides)]:
    avant = len(dataset)
    dataset = dataset.merge(source.dropna(subset=["iso3"]), on=["iso3", "year"], how="left")
    print(f"+ {nom:12} {avant} -> {len(dataset)} lignes")
    assert len(dataset) == avant, f"fan-out lors de la jointure {nom}"

doublons = dataset.duplicated(["iso3", "year", "crop"]).sum()
print(f"\ndoublons sur iso3 + year + crop : {doublons}")
assert doublons == 0, "la clé finale n'est pas unique"

base rendement  22679 lignes
+ température  22679 -> 22679 lignes
+ pluie        22679 -> 22679 lignes
+ pesticides   22679 -> 22679 lignes

doublons sur iso3 + year + crop : 0


# Contrôles du dataset

In [10]:
print(f"lignes    : {len(dataset)}")
print(f"pays      : {dataset['iso3'].nunique()}")
print(f"cultures  : {dataset['crop'].nunique()}")
print(f"années    : {dataset['year'].min()} - {dataset['year'].max()} "
      f"({dataset['year'].nunique()} distinctes)")
print(f"colonnes  : {list(dataset.columns)}")
print()
print(dataset.dtypes.to_string())

lignes    : 22679
pays      : 168
cultures  : 10
années    : 1990 - 2013 (24 distinctes)
colonnes  : ['iso3', 'area', 'year', 'crop', 'yield_t_ha', 'avg_temp', 'rain_mm', 'pesticides_t']

iso3             object
area             object
year              int64
crop             object
yield_t_ha      float64
avg_temp        float64
rain_mm         float64
pesticides_t    float64


In [11]:
print("valeurs manquantes (%) :")
print((dataset.isna().mean() * 100).round(1).to_string())

complet = dataset.dropna()
print(f"\nlignes complètes : {len(complet)} ({len(complet)/len(dataset):.1%}) "
      f"pour {complet['iso3'].nunique()} pays")

valeurs manquantes (%) :
iso3             0.0
area             0.0
year             0.0
crop             0.0
yield_t_ha       0.0
avg_temp        19.5
rain_mm          5.0
pesticides_t    13.1

lignes complètes : 15664 (69.1%) pour 117 pays


In [12]:
dataset[["year", "yield_t_ha", "avg_temp", "rain_mm", "pesticides_t"]].describe().round(2)

,year,yield_t_ha,avg_temp,rain_mm,pesticides_t
count,22679.00,22679.00,18264.00,21549.00,19704.00
mean,2001.72,6.64,19.82,1210.07,27939.03
std,6.87,7.47,6.99,768.29,140853.28
min,1990.00,0.00,-3.37,51.00,0.00
25%,1996.00,1.71,14.78,618.00,160.33
50%,2002.00,3.70,20.88,1130.00,1695.71
75%,2008.00,8.93,26.31,1712.00,10881.83
max,2013.00,55.49,30.42,3240.00,1806000.00


**Observations :**

- Granularité et clé conformes : `iso3 + year + crop` unique, aucun fan-out sur les trois jointures.
- Unités cohérentes : rendement en t/ha, température en °C, pluie en mm/an, pesticides en tonnes.
- Les manquants de `avg_temp` viennent d'une absence de couverture pays, pas de trous dans les séries : les sources n'ont aucun `NaN` de température après 1900.

# Sauvegarde

Toutes les lignes sont conservées, valeurs manquantes comprises : le fichier reste le
reflet des sources. Les valeurs manquantes sont conservées afin de laisser leur
traitement à l'étape de préparation des données.

Le fichier n'est pas versionné, comme les données sources. Il se reconstruit en
réexécutant ce notebook.

In [13]:
PATHS.data_processed.mkdir(parents=True, exist_ok=True)
chemin_sortie = PATHS.data_processed / f"crop_yield_prediction_{DEBUT}_{FIN}.csv"

dataset.to_csv(chemin_sortie, index=False)
# chemin relatif : aucun chemin absolu de la machine dans les outputs versionnés
print(f"écrit : {chemin_sortie.relative_to(PATHS.root)}")
print(f"taille : {chemin_sortie.stat().st_size / 1024**2:.1f} Mo")

écrit : data/processed/crop_yield_prediction_1990_2013.csv
taille : 1.2 Mo


In [14]:
# Relecture : le fichier écrit correspond-il au dataset en mémoire ?
relu = pd.read_csv(chemin_sortie)

print(f"lignes    : {len(relu)} (attendu {len(dataset)})")
print(f"colonnes  : {list(relu.columns) == list(dataset.columns)}")
print(f"doublons sur la clé : {relu.duplicated(['iso3', 'year', 'crop']).sum()}")
print(f"manquants identiques : {relu.isna().sum().equals(dataset.isna().sum())}")
print()
relu.head()

lignes    : 22679 (attendu 22679)
colonnes  : True
doublons sur la clé : 0
manquants identiques : True



,iso3,area,year,crop,yield_t_ha,avg_temp,rain_mm,pesticides_t
0,AFG,Afghanistan,1990,Maize,1.7582,15.45,327.0,NaN
1,AFG,Afghanistan,1991,Maize,1.6800,14.57,327.0,NaN
2,AFG,Afghanistan,1992,Maize,1.5000,14.35,327.0,NaN
3,AFG,Afghanistan,1993,Maize,1.6786,14.96,327.0,NaN
4,AFG,Afghanistan,1994,Maize,1.6667,14.94,327.0,NaN


**Observations :**

- Le fichier relu est identique au dataset en mémoire : lignes, colonnes, clé et manquants.
- La période pourra être réévaluée lors de la validation temporelle.